# Module 5 Demo (High-Level Overview) — Configuration Management, Reproducibility & Experiment Tracking

Imagine your manager asks on Friday afternoon:
> *"Last Tuesday we got an ROC-AUC of 0.86 — which features, hyperparameters, and dataset version produced that model?"*

If you only have overwritten notebook cells, you can't answer! Today's final module introduces the **3 pillars of disciplined ML engineering**:
1. **Configuration Management (YAML)**: Keep hyperparameters and paths in `configs/train_config.yaml`, not buried inside code.
2. **Reproducibility**: Fix random seeds, record library versions, and hash the input dataset so anyone can recreate your exact score.
3. **Experiment Tracking**: Log every run's parameters, metrics, and artifact path into a structured ledger (we'll build a transparent local JSONL tracker first, and then show how **MLflow** maps 1-to-1 to the exact same concepts!).


In [ ]:
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import random
import uuid

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import yaml
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


## Part 1: Configuration Management with `configs/train_config.yaml`

Instead of editing 6 places in Python code to try a new model or depth, we read one clean YAML file.


In [ ]:
CONFIG_PATH = Path("configs/train_config.yaml")
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print(f"Loaded config from: {CONFIG_PATH}")
print(json.dumps(config, indent=2))


## Part 2: Reproducibility — Seeds, Data Fingerprints & Environment Snapshots

True reproducibility needs **four ingredients**:
1. Fixed random seed (`random`, `numpy`, `sklearn` `random_state`)
2. Data fingerprint (SHA-256 hash of input files)
3. Exact configuration snapshot (the YAML dictionary used for that run)
4. Software environment snapshot (Python & scikit-learn versions — which our Docker container locks!)


In [ ]:
def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


def file_sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()[:12]


# Prove reproducibility: two independent runs with seed=42 produce bit-identical predictions!
def quick_rf_probs(seed: int) -> np.ndarray:
    set_global_seed(seed)
    X_syn = np.random.randn(200, 5)
    y_syn = (X_syn[:, 0] + X_syn[:, 1] > 0).astype(int)
    model = RandomForestClassifier(n_estimators=50, random_state=seed).fit(X_syn[:150], y_syn[:150])
    return model.predict_proba(X_syn[150:])[:, 1]


run_a = quick_rf_probs(seed=42)
run_b = quick_rf_probs(seed=42)
run_c = quick_rf_probs(seed=99)

print("Run A (seed=42) vs Run B (seed=42) exact match?:", np.array_equal(run_a, run_b))
print("Run A (seed=42) vs Run C (seed=99) exact match?:", np.array_equal(run_a, run_c))


## Part 3: Lightweight Experiment Tracking (and How It Maps to MLflow / W&B)

Every experiment tracking platform (MLflow, Weights & Biases, Vertex AI Experiments) logs four things per run:
- **Metadata**: `run_id`, `timestamp`, `data_hash`, `sklearn_version`
- **Parameters**: model type, hyperparameters, `random_seed`, `test_size`
- **Metrics**: `cv_roc_auc`, `test_roc_auc`, `test_f1`
- **Artifacts**: path to saved `.joblib` model file

Let's run **3 controlled experiments** using config overrides and log them cleanly into `artifacts/demo_experiments.jsonl`!


In [ ]:
def prepare_data(cfg: dict) -> tuple[pd.DataFrame, dict]:
    cust_path = Path(cfg["data"]["customers_csv"])
    usage_path = Path(cfg["data"]["usage_json"])

    cust = pd.read_csv(cust_path)
    with open(usage_path, "r", encoding="utf-8") as f:
        usage = pd.DataFrame(json.load(f))

    cust["plan_tier"] = cust["plan_tier"].apply(lambda x: x.strip().lower() if isinstance(x, str) else x)
    cust["country"] = cust["country"].apply(lambda x: x.strip().upper() if isinstance(x, str) else x)
    df = cust.drop_duplicates(subset=["customer_id"], keep="first").merge(usage, on="customer_id", how="left")

    df["charges_per_login"] = df["monthly_charges"] / (df["login_days_30d"] + 1.0)
    df["tickets_per_tenure_month"] = df["support_tickets_30d"] / (df["tenure_months"] + 1.0)
    df["watch_hours_per_login"] = df["watch_hours_30d"] / (df["login_days_30d"] + 1.0)
    df["is_new_customer"] = (df["tenure_months"] <= 3).astype(int)

    fingerprints = {
        "customers_sha256": file_sha256(cust_path),
        "usage_sha256": file_sha256(usage_path),
    }
    return df, fingerprints


def build_estimator(model_type: str, params: dict, seed: int):
    if model_type == "logistic_regression":
        return LogisticRegression(random_state=seed, **params)
    if model_type == "random_forest":
        return RandomForestClassifier(random_state=seed, **params)
    if model_type == "gradient_boosting":
        return GradientBoostingClassifier(random_state=seed, **params)
    raise ValueError(f"Unsupported model_type: {model_type}")


def run_tracked_experiment(
    cfg: dict,
    experiment_name: str,
    model_type: str,
    model_params: dict,
    log_file: Path,
) -> dict:
    seed = cfg["reproducibility"]["random_seed"]
    set_global_seed(seed)

    df, fingerprints = prepare_data(cfg)
    num_cols = cfg["features"]["numeric_columns"]
    cat_cols = cfg["features"]["categorical_columns"]
    target = cfg["data"]["target_column"]

    X = df[num_cols + cat_cols]
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=cfg["reproducibility"]["test_size"], random_state=seed, stratify=y
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
            (
                "cat",
                Pipeline(
                    [
                        ("imp", SimpleImputer(strategy="most_frequent")),
                        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                    ]
                ),
                cat_cols,
            ),
        ]
    )
    pipe = Pipeline([("prep", preprocessor), ("clf", build_estimator(model_type, model_params, seed))])

    cv = StratifiedKFold(n_splits=cfg["reproducibility"]["cv_folds"], shuffle=True, random_state=seed)
    cv_auc = float(cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc").mean())

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    run_id = uuid.uuid4().hex[:8]
    artifact_path = Path(cfg["artifacts"]["output_dir"]) / f"model_{run_id}.joblib"
    artifact_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(pipe, artifact_path)

    record = {
        "run_id": run_id,
        "experiment_name": experiment_name,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "model_type": model_type,
        "params": model_params,
        "random_seed": seed,
        "sklearn_version": sklearn.__version__,
        "data_fingerprints": fingerprints,
        "cv_roc_auc": round(cv_auc, 4),
        "test_roc_auc": round(float(roc_auc_score(y_test, y_prob)), 4),
        "test_f1": round(float(f1_score(y_test, y_pred)), 4),
        "artifact_path": str(artifact_path),
    }

    log_file.parent.mkdir(parents=True, exist_ok=True)
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")

    return record


# Start a clean demo log for this notebook run
demo_log = Path("artifacts/demo_experiments.jsonl")
if demo_log.exists():
    demo_log.unlink()

experiment_grid = [
    ("baseline_logreg", "logistic_regression", {"C": 1.0, "max_iter": 1000, "class_weight": "balanced"}),
    ("rf_depth6", "random_forest", {"n_estimators": 120, "max_depth": 6, "min_samples_leaf": 4, "class_weight": "balanced"}),
    ("gb_lr05", "gradient_boosting", {"n_estimators": 100, "learning_rate": 0.05, "max_depth": 3}),
]

for name, mtype, mparams in experiment_grid:
    rec = run_tracked_experiment(config, name, mtype, mparams, demo_log)
    print(f"Logged run {rec['run_id']} ({name:<16}) -> CV AUC={rec['cv_roc_auc']:.4f} | Test AUC={rec['test_roc_auc']:.4f} | F1={rec['test_f1']:.4f}")


In [ ]:
# Load the experiment ledger and visualize the leaderboard!
records = [json.loads(line) for line in demo_log.read_text(encoding="utf-8").splitlines()]
leaderboard = pd.DataFrame(records)[
    ["run_id", "experiment_name", "model_type", "cv_roc_auc", "test_roc_auc", "test_f1", "artifact_path"]
].sort_values("cv_roc_auc", ascending=False)

display(leaderboard)

ax = leaderboard.set_index("experiment_name")[["cv_roc_auc", "test_roc_auc", "test_f1"]].plot(
    kind="bar", figsize=(8, 4), rot=0, ylim=(0.5, 1.0)
)
ax.set_title("StreamPulse Experiment Tracker Leaderboard")
ax.set_ylabel("Score")
plt.tight_layout()
plt.show()


## Part 4: How This Maps 1-to-1 to MLflow in Production

Notice how our `run_tracked_experiment()` already captures the exact same structure used by **MLflow**! When you join a company running an MLflow server, the code looks almost identical:

```python
import mlflow
import mlflow.sklearn

mlflow.set_experiment("streampulse-churn")
with mlflow.start_run(run_name="rf_depth6"):
    mlflow.log_params(model_params)
    mlflow.log_param("data_sha256", fingerprints["customers_sha256"])
    mlflow.log_metrics({"cv_roc_auc": cv_auc, "test_roc_auc": test_auc, "test_f1": test_f1})
    mlflow.sklearn.log_model(pipe, artifact_path="model")
```

**Next Up:** Put everything from Day 6 together in **`09_capstone_end_to_end_pipeline_starter.ipynb`**!
